In [1]:
import pandas as pd
import numpy as np

In [2]:
events = pd.read_parquet(f"data/events.parquet")
races = pd.read_parquet(f"data/races.parquet")
runners = pd.read_parquet(f"data/runners.parquet")

In [3]:
fbase = "output/data"

# Pre-prep data

## Location Names

In [4]:
lmap = {
    "Aint": "Aintree",
    "Bang": "Bangor On Dee",
    "Bangor": "Bangor On Dee",
    "Bangor on Dee": "Bangor On Dee",
    "Bev": "Beverly",
    "Brig": "Brighton",
    "Carl": "Carlisle",
    "Cart": "Cartmel",
    "Catt": "Cattertick",
    "ChelmC": "Chelmsford City",
    "ChelmC PA": "Chelmsford City",
    "Chelt": "Cheltenham",
    "Chep": "Chepstow",
    "Chest": "Chester",
    "Donc": "Doncaster",
    "Epsm": "Epsom",
    "Extr": "Exeter",
    "Fake": "Fakenham",
    "FfosL": "Ffoss Las",
    "Font": "Fontwell",
    "GLghs": "Chelmsford City",
    "Good": "Goodwood",
    "Ham": "Hamilton",
    "Hayd": "Haydock",
    "Here": "Hereford",
    "Hex": "Hexham",
    "Hunt": "Huntingdon",
    "Kemp": "Kempton",
    "Leic": "Leicester",
    "Ling": "Lingfield",
    "Lud": "Ludlow",
    "MrktR": "Market Rasen",
    "Muss": "Musselburgh",
    "Newb": "Newbury",
    "Newc": "Newcastle",
    "Newm": "Newmarket",
    "Newt": "Newton Abbot",
    "Nott": "Nottingham",
    "Plump": "Plumpton",
    "Ponte": "Pontefract",
    "Redc": "Redcar",
    "Salis": "Salisbury",
    "Sand": "Sandown",
    "Sedge": "Sedgefield",
    "Sthl": "Southwell",
    "Strat": "Stratford",
    "Taun": "Taunton",
    "Uttox": "Uttoxeter",
    "Warw": "Warwick",
    "Weth": "Wetherby",
    "Winc": "Wincanton",
    "Wind": "Windsor",
    "Wolv": "Wolverhampton",
    "Worc": "Worcester",
    "Yarm": "Yarmouth"
}
events.location = events.location.replace(lmap)

## Normalised Probabilities

In [5]:
book_pct = runners.groupby("race_id").implied_odds.sum()
runners["normalised_odds"] = runners.implied_odds/runners.race_id.map(book_pct)

## Test Statistics

In [6]:
runners["returns"] = runners.bsp * runners.win_lose
runners["brier"]   = np.square(runners.normalised_odds - runners.win_lose)
runners["log_loss"] = -np.log(runners.normalised_odds)*runners.win_lose

## Bsp Binning Range

In [7]:
ticks = np.concat((
    np.arange(1, 2, 0.01),
    np.arange(2,3,0.02),
    np.arange(3,4,0.05),
    np.arange(4,6,0.1),
    np.arange(6,10,0.2),
    np.arange(10,20,0.5),
    np.arange(20,30,1),
    np.arange(30,50,2),
    np.arange(50,100,5),
    np.arange(100,1000,10),
    [1001]
))
#ticks=1000

# Favourite-Longshot Bias

## General

In [8]:
g = runners.groupby(pd.cut(runners.bsp, bins=ticks))

In [9]:
df = pd.DataFrame({"mean": g.win_lose.mean(), "count": g.size(), "volume": g.pptradedvol.sum()})

In [10]:
df = df.rolling(5).mean()
df["l"] = df.index.categories.left
df["r"] = df.index.categories.right
df.index=df.r
df["width"] = df.r-df.l

In [11]:
df.to_csv(f"{fbase}/all-fl.csv")

## Groups

per race stats

In [12]:
rg = runners.groupby("race_id")

nraces = races.assign(
    ppvol=rg.pptradedvol.sum(),
    nrunners=rg.size(),
    overround=rg.implied_odds.sum()
)

add groups that are needed

In [13]:
df = runners.join(nraces, on="race_id").join(events, on="event_id")

In [14]:
cats = {
    "race_no": ("race_no", [1,2,3,4,5,6,7,8]),
    "distance_f": ("distance_f", df.distance_f.value_counts().head(10).index),
    "hour": (df.race_time.dt.hour, np.arange(12,20,1)),
    "race_type": ("race_type", df.race_type.value_counts().head(10).index),
    "location": ("location", df.location.value_counts().head(10).index),
    "nrunners": ("nrunners", df.nrunners.value_counts().head(10).index),
    "ppvol": (pd.qcut(df.ppvol, q=10), slice(None)),
    "overround": (pd.qcut(df.overround, q=10), slice(None))
}

In [15]:
for c in cats:
    for a, v in [("mean", "win_lose"), ("count", "win_lose"), ("sum", "pptradedvol")]:
        pt = df.pivot_table(
            index=pd.cut(df.bsp, bins=ticks),
            columns=cats[c][0],
            values=v,
            aggfunc=a
        ).loc[:, cats[c][1]]
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        pt=pt.rolling(5).mean()
        l = pt.index.categories.left
        r = pt.index.categories.right
        pt.insert(0, 'width', r-l)
        pt.index=l
        pt.to_csv(f"{fbase}/{c}-fl-{a}.csv")

## Test Statistics

In [16]:
for c in cats:
    b = df.groupby(cats[c][0]).brier.mean()
    l = df.groupby(["race_id", cats[c][0]]).log_loss.max().groupby(level=1).mean()

    f = pd.DataFrame({"brier": b, "log_loss": l, "count": df.groupby(cats[c][0]).size()})
    f = f.loc[cats[c][1], :]
    f = f.sort_index()
    f.to_csv(f"{fbase}/{c}-acc.csv")

In [17]:
b = df.groupby(pd.qcut(df.ppvol, q=10)).brier.mean()

In [18]:
l = df.groupby(["race_id", pd.qcut(df.ppvol, q=10)]).log_loss.max().groupby(level=1).mean()

In [19]:
f = pd.DataFrame({"brier": b, "log_loss": l})

In [20]:
f

,brier,log_loss
ppvol,,
"(-0.001, 204639.938]",0.090479,1.835907
"(204639.938, 266895.75]",0.088474,1.766805
"(266895.75, 327783.531]",0.086166,1.804845
"(327783.531, 386451.0]",0.084944,1.843476
"(386451.0, 445733.75]",0.084928,1.840925
"(445733.75, 507527.0]",0.085633,1.813884
"(507527.0, 579044.438]",0.086610,1.770688
"(579044.438, 673881.938]",0.086763,1.696709
"(673881.938, 834048.0]",0.086144,1.624130


In [21]:
f[

_IncompleteInputError: incomplete input (26326615.py, line 1)